# Auralee Viability Evaluation Notebook

Run: `gcloud auth application-default login` first so Firestore client can read.


In [ ]:
from datetime import datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo
from google.cloud import firestore
import pandas as pd
import plotly.express as px

EVALUATION_DAYS = 14  # Set to the completed uninterrupted window: 14–30 days
if not 14 <= EVALUATION_DAYS <= 30:
    raise ValueError('EVALUATION_DAYS must be between 14 and 30')
MARKET_TZ = ZoneInfo('America/New_York')
MIN_EVALUATED_SAMPLES = 30
MIN_EVALUATION_COVERAGE = 0.80
MIN_PRICE_SAMPLES_PER_CLASS = 30
MIN_PRICE_COVERAGE = 0.50
MIN_COLLECTION_ATTEMPTS_PER_SOURCE = 20
EXPECTED_RUNS_PER_SOURCE = EVALUATION_DAYS * 24  # both P1 scrapers are hourly
MIN_RUN_SCHEDULE_COVERAGE = 0.80
P1_SOURCES = {'hn', 'reuters'}  # MarketWatch still uses the legacy reuters label
db = firestore.Client(project="auralee-api-server")
cutoff = datetime.now(timezone.utc) - timedelta(days=EVALUATION_DAYS)

def safe_coverage(sample_n, eligible_n):
    return sample_n / eligible_n if eligible_n else float('nan')

def format_or_na(value, format_spec):
    return format(value, format_spec) if pd.notna(value) else 'N/A'


## Load articles (evaluation window)


In [ ]:
ARTICLE_COLUMNS = [
    'published_at', 'processed_at', 'source', 'sanity_check', 'eval_score',
    'sentiment', 'tickers', 'gemini_meta',
]
article_rows = [
    d.to_dict() for d in
    db.collection('articles').where('processed_at', '>=', cutoff.isoformat()).stream()
]
articles = pd.DataFrame(article_rows)
for column in ARTICLE_COLUMNS:
    if column not in articles:
        articles[column] = pd.Series(index=articles.index, dtype='object')
for column in ['published_at', 'processed_at']:
    articles[column] = pd.to_datetime(articles[column], errors='coerce', utc=True)
print(f"Loaded {len(articles)} articles")


## 1. Volume by source per day


In [ ]:
dated_articles = articles.dropna(subset=['published_at', 'source']).copy()
if dated_articles.empty:
    daily = pd.DataFrame()
    print('No dated articles available for the volume chart.')
else:
    dated_articles['day'] = dated_articles['published_at'].dt.date
    daily = dated_articles.groupby(['day', 'source']).size().unstack(fill_value=0)
    px.line(daily, title='Articles/day by source').show()
print(f'Volume sample: n={len(articles)} articles in a {EVALUATION_DAYS}-day window')


## 2. M2 sanity + M3 judge distributions


In [ ]:
m2_values = articles['sanity_check'].apply(
    lambda value: value.get('ticker_precision_pass') if isinstance(value, dict) else None
)
m2_sample = m2_values.dropna().astype(bool)
m2_rate = m2_sample.mean() if len(m2_sample) else float('nan')
m2_coverage = safe_coverage(len(m2_sample), len(articles))
print(
    f'M2 precision rate: {format_or_na(m2_rate, ".3f")}  '
    f'(n={len(m2_sample)}, coverage={format_or_na(m2_coverage, ".1%")})'
)

m3_values = pd.to_numeric(articles['eval_score'].apply(
    lambda value: value.get('score') if isinstance(value, dict) else None
), errors='coerce')
m3_scores = m3_values.dropna()
m3_avg = m3_scores.mean() if len(m3_scores) else float('nan')
m3_coverage = safe_coverage(len(m3_scores), len(articles))
print(
    f'M3 avg score: {format_or_na(m3_avg, ".2f")}  '
    f'(n={len(m3_scores)}, coverage={format_or_na(m3_coverage, ".1%")})'
)
if len(m3_scores):
    px.histogram(m3_scores, nbins=20, title='M3 judge score distribution').show()


## 3. M2 <-> M3 disagreement

The denominator contains only articles that completed both M2 and M3.


In [ ]:
cross_mask = m2_values.notna() & m3_values.notna()
cross_evaluated = pd.DataFrame({
    'm2_pass': m2_values.loc[cross_mask].astype(bool),
    'm3_score': m3_values.loc[cross_mask],
})
disagreement = (
    (cross_evaluated['m2_pass'] & (cross_evaluated['m3_score'] < 4))
    | (~cross_evaluated['m2_pass'] & (cross_evaluated['m3_score'] > 7))
)
disagree_rate = disagreement.mean() if len(disagreement) else float('nan')
cross_coverage = safe_coverage(len(cross_evaluated), len(articles))
print(
    f'Disagreement rate: {format_or_na(disagree_rate, ".3f")} '
    f'(disagreements={int(disagreement.sum())}, n={len(cross_evaluated)}, '
    f'coverage={format_or_na(cross_coverage, ".1%")})'
)


## 4. Sentiment distribution


In [ ]:
sentiment_scores = pd.to_numeric(articles['sentiment'].apply(
    lambda value: value.get('score') if isinstance(value, dict) else None
), errors='coerce')
sentiment_plot = pd.DataFrame({
    'sentiment_score': sentiment_scores,
    'source': articles['source'],
}).dropna(subset=['sentiment_score'])
if sentiment_plot.empty:
    print('No sentiment scores available.')
else:
    px.histogram(
        sentiment_plot, x='sentiment_score', color='source',
        title='Sentiment score distribution',
    ).show()


## 5. Price reaction (CORE hypothesis)

Conservative daily-bar protocol (first ticker only): pre-market uses the immediately preceding weekday close to same-day close; after-hours uses same-day close to the immediately following weekday close; weekends use Friday-to-Monday when both bars exist. Regular-session articles and every unknown weekday gap are excluded, so a missing price refresh cannot be mistaken for a holiday. Publication times are converted to the New York market timezone. P1 should replace this conservative proxy with an official NYSE session calendar.


In [ ]:
def daily_close(ticker, trading_date):
    try:
        snapshot = (
            db.collection('prices').document(ticker).collection('daily')
            .document(trading_date.strftime('%Y%m%d')).get()
        )
    except Exception:
        return None
    if not snapshot.exists:
        return None
    close = pd.to_numeric((snapshot.to_dict() or {}).get('close'), errors='coerce')
    return None if pd.isna(close) else float(close)

def previous_weekday(trading_date):
    candidate = trading_date - timedelta(days=1)
    while candidate.weekday() >= 5:
        candidate -= timedelta(days=1)
    return candidate

def next_weekday(trading_date):
    candidate = trading_date + timedelta(days=1)
    while candidate.weekday() >= 5:
        candidate += timedelta(days=1)
    return candidate

def session_aligned_observation(row):
    tickers = row.get('tickers')
    published_at = row.get('published_at')
    if not isinstance(tickers, (list, tuple)) or not tickers or pd.isna(published_at):
        return None, 'invalid_article'
    ticker = str(tickers[0]).upper()  # PoC: one observation per article
    published_local = published_at.tz_convert(MARKET_TZ)
    published_date = published_local.date()
    published_time = published_local.time()
    same_day_close = daily_close(ticker, published_date)

    if published_date.weekday() < 5 and same_day_close is None:
        return None, 'missing_or_closed_publication_bar'
    if same_day_close is not None and time(9, 30) <= published_time < time(16):
        return None, 'intraday_unsupported'
    if same_day_close is not None and published_time < time(9, 30):
        baseline_close = daily_close(ticker, previous_weekday(published_date))
        target_close = same_day_close
    elif same_day_close is not None:
        baseline_close = same_day_close
        target_close = daily_close(ticker, next_weekday(published_date))
    else:  # weekend; weekdays with a missing/closed bar returned above
        baseline_close = daily_close(ticker, previous_weekday(published_date))
        target_close = daily_close(ticker, next_weekday(published_date))
    if baseline_close in (None, 0) or target_close is None:
        return None, 'missing_adjacent_bar'
    return (target_close - baseline_close) / baseline_close, 'matched'

def is_price_eligible(row):
    tickers = row.get('tickers')
    sentiment = row.get('sentiment')
    return (
        isinstance(tickers, (list, tuple)) and bool(tickers)
        and pd.notna(row.get('published_at'))
        and isinstance(sentiment, dict)
        and sentiment.get('label') in {'bullish', 'bearish'}
    )

if len(articles):
    price_eligible_mask = articles.apply(is_price_eligible, axis=1)
    price_observations = articles.apply(session_aligned_observation, axis=1)
    articles['ndr'] = price_observations.apply(lambda value: value[0])
    articles['price_match_status'] = price_observations.apply(lambda value: value[1])
else:
    price_eligible_mask = pd.Series(index=articles.index, dtype='bool')
    articles['ndr'] = pd.Series(index=articles.index, dtype='float64')
    articles['price_match_status'] = pd.Series(index=articles.index, dtype='object')
analysis = articles.loc[price_eligible_mask & articles['ndr'].notna()].copy()
analysis['label'] = analysis['sentiment'].apply(
    lambda value: value.get('label') if isinstance(value, dict) else None
)
price_eligible_n = int(price_eligible_mask.sum())
price_sample_n = len(analysis)
price_bullish_n = int((analysis['label'] == 'bullish').sum())
price_bearish_n = int((analysis['label'] == 'bearish').sum())
price_coverage = safe_coverage(price_sample_n, price_eligible_n)
print(
    f'Price coverage: {price_sample_n}/{price_eligible_n} eligible articles '
    f'({format_or_na(price_coverage, ".1%")}); '
    f'bullish={price_bullish_n}, bearish={price_bearish_n}; '
    'session-aligned daily-bar proxy; first ticker only'
)
if analysis.empty:
    print('No matched price-reaction samples available.')
else:
    print(analysis.groupby('label')['ndr'].agg(['mean', 'std', 'count']))
print(articles.loc[price_eligible_mask, 'price_match_status'].value_counts(dropna=False))
print('Regular-session publications are excluded; add intraday bars before evaluating them.')


## 6. Collection delivery analysis (runs collection)

P1 collection success covers Hacker News and MarketWatch (legacy `reuters` label) and is weighted by candidate outcomes: `(ingested + duplicate) / attempted`. WSJ is a separate desktop spike, not a cloud-pipeline gate.


In [ ]:
RUN_COLUMNS = [
    'kind', 'source', 'status', 'articles_attempted',
    'articles_ingested', 'articles_skipped_dup',
]
run_rows = [d.to_dict() for d in
    db.collection('runs').where('started_at', '>=', cutoff.isoformat()).stream()]
runs = pd.DataFrame(run_rows)
for column in RUN_COLUMNS:
    if column not in runs:
        runs[column] = pd.Series(index=runs.index, dtype='object')
for column in ['articles_attempted', 'articles_ingested', 'articles_skipped_dup']:
    runs[column] = pd.to_numeric(runs[column], errors='coerce').fillna(0)
runs['run_success'] = runs['status'].eq('success')
if runs.empty:
    print('No runs available for error analysis.')
else:
    print(runs.groupby(['kind', 'source'], dropna=False)['run_success'].agg(['count', 'mean']))

collection_runs = runs[(runs['kind'] == 'scrape') & runs['source'].isin(P1_SOURCES)].copy()
successful_outcomes = (
    collection_runs['articles_ingested'] + collection_runs['articles_skipped_dup']
)
collection_runs['successful_outcomes'] = pd.concat(
    [successful_outcomes, collection_runs['articles_attempted']], axis=1
).min(axis=1)
source_delivery = collection_runs.groupby('source').agg(
    runs=('run_success', 'size'),
    run_success_rate=('run_success', 'mean'),
    attempted=('articles_attempted', 'sum'),
    successful=('successful_outcomes', 'sum'),
).reindex(sorted(P1_SOURCES))
for column in ['runs', 'attempted', 'successful']:
    source_delivery[column] = source_delivery[column].fillna(0).astype(int)
source_delivery['candidate_success_rate'] = source_delivery.apply(
    lambda row: safe_coverage(row['successful'], row['attempted']), axis=1
)
collection_attempted = int(source_delivery['attempted'].sum())
collection_successful = int(source_delivery['successful'].sum())
min_source_attempted = int(source_delivery['attempted'].min())
min_source_runs = int(source_delivery['runs'].min())
source_coverage = safe_coverage(int((source_delivery['runs'] > 0).sum()), len(P1_SOURCES))
run_schedule_coverage = safe_coverage(min_source_runs, EXPECTED_RUNS_PER_SOURCE)
collection_candidate_success_rate = source_delivery['candidate_success_rate'].min(skipna=False)
collection_run_success_rate = source_delivery['run_success_rate'].min(skipna=False)
print(
    f'P1 candidate success (worst source): '
    f'{format_or_na(collection_candidate_success_rate, ".1%")} '
    f'(successful={collection_successful}, attempted={collection_attempted}, '
    f'runs={len(collection_runs)})'
)
print(
    f'P1 run success (worst source): {format_or_na(collection_run_success_rate, ".1%")} '
    f'(minimum runs/source={min_source_runs}/{EXPECTED_RUNS_PER_SOURCE}, schedule coverage='
    f'{format_or_na(run_schedule_coverage, ".1%")}, source coverage='
    f'{format_or_na(source_coverage, ".1%")})'
)
print(source_delivery)
print('Duplicates count as successful fetches; a partial run is not counted wholesale as success.')


## 7. Cost tracking


In [ ]:
articles['cost'] = pd.to_numeric(articles['gemini_meta'].apply(
    lambda value: value.get('cost_usd') if isinstance(value, dict) else None
), errors='coerce')
cost_sample_n = int(articles['cost'].notna().sum())
cost_coverage = safe_coverage(cost_sample_n, len(articles))
total_cost = articles['cost'].sum(min_count=1)
avg_cost = articles['cost'].mean()
total_cost_display = f'${total_cost:.2f}' if pd.notna(total_cost) else 'N/A'
avg_cost_display = f'${avg_cost:.4f}' if pd.notna(avg_cost) else 'N/A'
print(
    f'{EVALUATION_DAYS}-day Gemini cost: {total_cost_display}, '
    f'avg {avg_cost_display}/article '
    f'(n={cost_sample_n}, coverage={format_or_na(cost_coverage, ".1%")})'
)


## 8. Viability Scorecard

A threshold can pass only when the metric is evaluable. Quality and cost metrics require at least 30 samples and 80% coverage; price signal requires at least 30 bullish and 30 bearish samples with 50% price coverage. Each P1 source needs at least 20 attempted candidates and 80% of its hourly runs; candidate and run success use the worse source, so one healthy source cannot hide another.


In [ ]:
vol = len(articles) / EVALUATION_DAYS
grouped = (
    analysis.groupby('label')['ndr'].mean()
    if len(analysis) else pd.Series(dtype='float64')
)
spread = (
    grouped['bullish'] - grouped['bearish']
    if {'bullish', 'bearish'}.issubset(grouped.index) else float('nan')
)

def metric_pass(value, predicate, evaluable):
    return bool(evaluable and pd.notna(value) and predicate(value))

m2_evaluable = len(m2_sample) >= MIN_EVALUATED_SAMPLES and m2_coverage >= MIN_EVALUATION_COVERAGE
m3_evaluable = len(m3_scores) >= MIN_EVALUATED_SAMPLES and m3_coverage >= MIN_EVALUATION_COVERAGE
cross_evaluable = (
    len(cross_evaluated) >= MIN_EVALUATED_SAMPLES
    and cross_coverage >= MIN_EVALUATION_COVERAGE
)
price_evaluable = (
    price_bullish_n >= MIN_PRICE_SAMPLES_PER_CLASS
    and price_bearish_n >= MIN_PRICE_SAMPLES_PER_CLASS
    and price_coverage >= MIN_PRICE_COVERAGE
)
cost_evaluable = cost_sample_n >= MIN_EVALUATED_SAMPLES and cost_coverage >= MIN_EVALUATION_COVERAGE
candidate_delivery_evaluable = (
    source_coverage == 1
    and min_source_attempted >= MIN_COLLECTION_ATTEMPTS_PER_SOURCE
)
run_delivery_evaluable = (
    source_coverage == 1
    and run_schedule_coverage >= MIN_RUN_SCHEDULE_COVERAGE
)

scorecard = pd.DataFrame([
    ('volume/day',           vol,              len(articles),        float('nan'), len(articles) > 0, metric_pass(vol, lambda v: v > 100, len(articles) > 0)),
    ('m2_precision_rate',    m2_rate,          len(m2_sample),       m2_coverage, m2_evaluable, metric_pass(m2_rate, lambda v: v > 0.90, m2_evaluable)),
    ('m3_avg_score',         m3_avg,           len(m3_scores),       m3_coverage, m3_evaluable, metric_pass(m3_avg, lambda v: v > 7.0, m3_evaluable)),
    ('m2_m3_disagree_rate',  disagree_rate,    len(cross_evaluated), cross_coverage, cross_evaluable, metric_pass(disagree_rate, lambda v: v < 0.05, cross_evaluable)),
    ('price_signal_spread',  spread,           price_sample_n,       price_coverage, price_evaluable, metric_pass(spread, lambda v: v > 0.01, price_evaluable)),
    ('avg_cost_per_article', avg_cost,         cost_sample_n,        cost_coverage, cost_evaluable, metric_pass(avg_cost, lambda v: v < 0.01, cost_evaluable)),
    ('candidate_success_rate', collection_candidate_success_rate, min_source_attempted, source_coverage, candidate_delivery_evaluable, metric_pass(collection_candidate_success_rate, lambda v: v > 0.80, candidate_delivery_evaluable)),
    ('source_run_success_rate', collection_run_success_rate, min_source_runs, run_schedule_coverage, run_delivery_evaluable, metric_pass(collection_run_success_rate, lambda v: v > 0.80, run_delivery_evaluable)),
], columns=['metric', 'value', 'n', 'coverage', 'evaluable', 'pass'])
scorecard
